In [1]:
#This notebook will serve as a baseline with the regular
#Single Agent logic implemented in gymnasium as was done in 
#MATLAB
#This will allow us to try certain concepts in a controlled
#environment rather than the more complex MARL environment


In [4]:
import gymnasium as gym
import numpy as np
import matplotlib
from typing import Optional
from gymnasium import spaces
import random
from gymnasium.utils import seeding


In [5]:
#This environment gives us insight into the basic stochastic knapsack problem that we need to solve in the multi-agent environment
#Basically it is the same problem as the multi-agent environment without the complexities of fairness and collisions
class SingleSatelliteEnv(gym.Env):

    def __init__(self,render_mode):
        self.np_random = np.random.default_rng()
        self.timestep=None
        #Contact plan in this case only contains weather condition information (as we assume all equal contact lengths)
        self.contact_plan=np.full((10,), None)
        self.satellite_remaining_data=None
        self.delivered_data=None
        self.Energy_Expended=None
        self.initial_data_volume=None
        self.num_of_contacts=None
        #Observation space is a (12,) array containing the timestep, remaining satellite data and the weather information
        self.observation_space = gym.spaces.Box(
    low=np.array([0.0]*10 + [0.0] + [0.0], dtype=np.float32),
    high=np.array([1.0]*10 + [1.0] + [10.0], dtype=np.float32),
    dtype=np.float32
)
        
        self.action_space = gym.spaces.Discrete(2)
    
    
    
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        #self.np_random, seed = seeding.np_random(seed)
        super().reset(seed=seed)
        #Initialize timestep to 0, the first contact
        #if seed is not None:
        #    self.np_random = np.random.default_rng(seed)
        self.timestep=0
        #
        self.contact_plan=self.np_random.uniform(low=0.0, high=1.0, size=(10,)).astype(np.float32)
        self.contact_plan= np.round(self.contact_plan*10)/10
        self.satellite_remaining_data=np.array(
        self.np_random.integers(5, 100) / 100.0, dtype=np.float32
    )
        self.delivered_data=0
        self.initial_data_volume=self.satellite_remaining_data
        self.Energy_Expended=0
        self.num_of_contacts=0
        observation = self._get_obs()
        info = self._get_info()
        return observation, info
    def updateDeliveryandEnergy(self,weather,length,remaining_data):
        delivered_packets=0
        excess_energy_expended=0
        random_sample=self.np_random.uniform(low=0.0, high=1.0, size=(length,)).astype(np.float32)
        #print("Contact Conditions")
        #print(random_sample)
        remaining_data=np.round(remaining_data)
        for i in range(0,length):
            #print("Remaining Data")
            #print(remaining_data)
            if remaining_data>=1:
                #print(random_sample[i])
                #print(weather)
                if random_sample[i]> weather:
                    delivered_packets=delivered_packets+1
                    remaining_data=remaining_data-1
                else:
                    excess_energy_expended=excess_energy_expended+1
            else:
                excess_energy_expended=excess_energy_expended+1
        #print("Contact Delivered Data")
        #print(delivered_packets)
        return delivered_packets,excess_energy_expended
    def step(self, action):
        if action==0:
            reward=0
        else:
            #print(self.satellite_remaining_data)
            self.num_of_contacts+=1
            data_remaining=self.satellite_remaining_data*100
            current_weather=self.contact_plan[self.timestep]
            current_delivered_data,current_energy_expenditure=self.updateDeliveryandEnergy(current_weather,10,data_remaining)
            init_data_volume=self.initial_data_volume
            if current_delivered_data>0:
                reward=(10/(init_data_volume*100))*current_delivered_data-5/(100*init_data_volume)*current_delivered_data*(current_energy_expenditure/(current_energy_expenditure+10))
            else:
                reward=-(5*current_energy_expenditure)/(init_data_volume*100)
            self.delivered_data=self.delivered_data+current_delivered_data
            self.Energy_Expended=self.Energy_Expended+current_energy_expenditure
            self.satellite_remaining_data=(self.satellite_remaining_data)*100-current_delivered_data
            self.satellite_remaining_data=self.satellite_remaining_data/100
        self.updateTimestep()
        terminated=False
        truncated=False
        if self.timestep>=10:
            terminated=True
            delivered_data=self.delivered_data
            init_data_volume=self.initial_data_volume*100
            energy_expended=self.Energy_Expended
            reward=reward+10*(delivered_data/(init_data_volume*100))-5*(energy_expended/100)
        elif self.satellite_remaining_data<=0:
            truncated=True
            #Episode Reward
            energy_expended=self.Energy_Expended
            number_of_contacts_used=self.num_of_contacts
            reward=reward+10-(energy_expended/(10*(number_of_contacts_used)))*5
        observation = self._get_obs()
        info = self._get_info()
        reward=float(reward)
        return observation, reward, terminated, truncated, info
    def _get_info(self):
      
        return {"contact plan":self.contact_plan, "Remaining data":self.satellite_remaining_data,"current timestep":self.timestep,"Energy Expended":self.Energy_Expended,"Delivered Data":self.delivered_data,"Initial Data":self.initial_data_volume,"Number of Contacts":self.num_of_contacts}
    def _get_obs(self):
       return np.concatenate([
        np.array(self.contact_plan, dtype=np.float32),
        np.array([self.satellite_remaining_data], dtype=np.float32),
        np.array([self.timestep], dtype=np.float32)
    ])
        #return {"contact plan": self.contact_plan, "Remaining data": self.satellite_remaining_data,"current timestep":np.array([float(self.timestep)], dtype=np.float32)}
    def updateTimestep(self):
        self.timestep=self.timestep+1
        return self.timestep
    def updateDeliveredPackets(self,new_deliveries):
        self.delivered_data=self.delivered_data+new_deliveries
    def getDeliveredPackets(self):
        return self.delivered_data
    def getEnergyExpenditure(self):
        return self.Energy_Expended
    def updateEnergyExpenditure(self,new_expend):
        self.Energy_Expended=self.Energy_Expended+new_expend
    def updateRemainingData(self,new_deliveries):
        self.satellite_remaining_data=(self.satellite_remaining_data)*100-current_delivered_data
        self.satellite_remaining_data=self.satellite_remaining_data/100
        

In [3]:
#Next need to produce training graphs and compare between pre-existing agents
from gymnasium.envs.registration import register
register(
    id="SingleSatelliteEnv",
    entry_point="Single_ag_Environment:SingleSatelliteEnv",  # Adjust the module path
)

# Test if the environment works
#env = gym.make("SingleSatelliteEnv")
#obs, info = env.reset()
#print("Custom Env Loaded Successfully!")